# Experiment: Case-001 Branch Review Packet Checker Gate

Objective: smoke-test the branch-review packet checker with synthetic rows only.

Success criteria:
- incomplete synthetic rows emit the not-ready public label;
- complete synthetic rows emit the ready-for-private-clinician-review public label;
- private-only fields are not present in public output.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "scripts" / "check_branch_review_packet.py").exists()
)
sys.path.insert(0, str(repo_root / "scripts"))

import check_branch_review_packet as checker  # noqa: E402

checker.REQUIRED_DOMAINS

## Plan

Use synthetic rows to test the two public states. The notebook does not use real patient records, private references, contact details, or clinician messages.


In [ ]:
def synthetic_row(domain: str, readiness_label: str) -> dict[str, str]:
    """Return one synthetic private packet row."""
    private_ref = (
        "synthetic-private-ref"
        if readiness_label == checker.READY_READINESS_LABEL
        else ""
    )
    return {
        "domain": domain,
        "readiness_label": readiness_label,
        "owner_label": "synthetic_owner",
        "private_record_ref": private_ref,
        "public_missing_label": f"missing_{domain}",
        "public_ready_label": f"ready_{domain}",
        "notes_private": "synthetic private note",
    }


incomplete_rows = [
    synthetic_row(domain, "missing_private_record")
    for domain in checker.REQUIRED_DOMAINS
]
incomplete_result = checker.evaluate_rows(incomplete_rows)

assert incomplete_result["current_label"] == checker.NOT_READY_LABEL
assert incomplete_result["ready_domains"] == []
assert incomplete_result["missing_domains"] == list(checker.REQUIRED_DOMAINS)
assert not checker.PRIVATE_ONLY_COLUMNS.intersection(incomplete_result)

incomplete_result

In [ ]:
complete_rows = [
    synthetic_row(domain, checker.READY_READINESS_LABEL)
    for domain in checker.REQUIRED_DOMAINS
]
complete_result = checker.evaluate_rows(complete_rows)

assert complete_result["current_label"] == checker.READY_LABEL
assert complete_result["ready_domains"] == list(checker.REQUIRED_DOMAINS)
assert complete_result["missing_domains"] == []
assert not checker.PRIVATE_ONLY_COLUMNS.intersection(complete_result)

complete_result

## Result

The synthetic smoke test keeps the checker in a public-safe role: it can classify packet readiness, but it cannot select therapy, screen trials, route samples, or expose private references.
